# Format processed data

This notebook takes the processed data from the DADA2, VSEARCH and MetaBEAT pipelines from three real datasets and one mock dataset and formats them into a tidy master long dataset and phyloseq object. It also gets metadata from WoRMs. For each dataset (n = 4) and denoising option (n = 3), we have the following six processed files.

For ASV information before taxonomic assignment:
- ASV counts
- ASV seqs

For taxonomic assignment:
- MetaBEAT 
- RDP
- MAPseq 
- BLAST (LCA condensed)
- VSEARCH (LCA condensed)

We have 84 different files (4 x 3 x 7) which need to be manipulated into one single data frame. We can create both long and phyloseq formats. 

Note: MetaBEAT files are formatteed differently and have to be cleaned before introduced to the script.

## Set-up

In [0]:
# Install packages
installed <- rownames(installed.packages())

ensure_cran <- function(pkgs, repos = "https://www.stats.bris.ac.uk/R/") {
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    install.packages(to_install, dependencies = TRUE, repos = repos)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

ensure_bioc <- function(pkgs) {
  if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
  }
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    BiocManager::install(to_install, ask = FALSE, update = FALSE)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

## CRAN packages
cran_pkgs <- c("Rcpp", "devtools", "optparse", "taxonomizr", "dplyr", "seqinr", "worrms", "tidyverse", "janitor", "ggtext")
ensure_cran(cran_pkgs)

## Bioconductor packages
bioc_pkgs <- c("Biostrings", "ShortRead", "dada2", "phyloseq", "microbiome")
ensure_bioc(bioc_pkgs)

In [0]:
%python
pip install biopython

## Create master long dataset

Let's import and merge all the data into one master long dataset.

In [0]:
%sh
python -u Scripts/10_create_master_dataset.py # parameters to change at top of script

In [0]:
# read in data in R we just created
master_long_df <- read.csv("Data/Processed/master_long_data.csv")

# set factor variables
factor_cols <- c("asv","sampleid","denoise_method","dataset","taxonomy_method","kingdom","phylum","class","order","family","genus","species")

factor_cols <- factor_cols[factor_cols %in% colnames(master_long_df)]
master_long_df[factor_cols] <- lapply(master_long_df[factor_cols], as.factor)

#rename columns for worms merging later
colnames(master_long_df) <- c("asv","sampleid","reads", "sequence", "denoise_method","dataset","taxonomy_method","kingdom_db","phylum_db","class_db","order_db","family_db","genus_db","species_db", "taxonomy")

# check expected variables are there
#str(master_long_df)
print(levels(master_long_df$taxonomy_method))
print(levels(master_long_df$denoise_method))
print(levels(master_long_df$dataset))

# create a unique fullID
master_long_df$fullID = paste(master_long_df$sampleid, master_long_df$denoise_method, master_long_df$taxonomy_method, master_long_df$dataset, sep="_")

And we need the visual detections to compare against.

In [0]:
visual_data <- read.csv(file = "Data/Raw/Traditional_Data/known_taxa_pipeline_comparisons.csv")
visual_types <- read.csv(file = "Data/Raw/Traditional_Data/visual_types.csv")

# remove genus from visual types
visual_types <- visual_types %>%
  select(-genus) 

visual_data$dataset <- as.factor(visual_data$dataset)

## Extracting information from WoRMS and other metadata

We often get random spaces or phrases we don’t want in our taxonomic names. We can use the janitor package to clean our taxonomic names.

We can also get all metadata associated to our taxa from the World Register of Marine Species (WoRMS). 

This next script does both of those jobs and outputs a new long dataset called `master_long_worms_df`. It is also saved in the Data/Processed folder.

In [0]:
source("Scripts/10b_tidy_WoRMS.R")

In [0]:
# make new cleaned column
visual_data <- visual_data %>%
  mutate(
    taxa_name_clean = janitor::make_clean_names(taxa, allow_dupes = TRUE),
    taxa_name_clean = str_remove(taxa_name_clean, "_spp$")
  )

# get worms info
visual_unique_species <- unique(visual_data$taxa_name_clean)
visual_worms_lookup <- map_df(visual_unique_species, get_worms_record_safe)
visual_data <- visual_data %>%
  left_join(visual_worms_lookup, by = c("taxa_name_clean" = "input_name"))

# run worms
visual_data

In [0]:
write.csv(visual_data, file = "Data/Processed/visual_data_tidy.csv")

Add whether we think something is a false positive, false negative, or menu species.

In [0]:
master_long_worms_df <- master_long_worms_df %>%
  left_join(
    visual_types,
    by = c("species", "dataset")
  ) %>%
  mutate(
    type = replace_na(type, "eDNA_only")
  )

## Explore and filter low abundance detections

Following the WFD methods, we remove any occurences that make up less than 0.01% of the community within a sample. This is to reduce the chances of false positives.

First, let's check that 0.001 is a suitable threshold.

In [0]:
master_long_worms_df <- master_long_worms_df %>%
  group_by(sampleid, taxonomy) %>%
  mutate(taxa_reads = sum(reads, na.rm = TRUE)) %>%
  group_by(sampleid) %>%
  mutate(
    total_reads = sum(reads, na.rm = TRUE),
    rel_abundance_per_sample = taxa_reads / total_reads
  ) %>%
  ungroup()

In [0]:
false_negative_taxa <- master_long_worms_df %>%
  filter(type == "false_negative") %>%
  distinct(taxa_name_final) %>%
  arrange(taxa_name_final)

false_negative_taxa

In [0]:
plot_df <- subset(
  master_long_worms_df,
  !type %in% c("eDNA_only", "uncertain")
)

plot_df$dataset <- factor(
  plot_df$dataset,
  levels = c("ringtrial_sean", "marchamley", "windermere_2017"),
  labels = c("Loch Insh", "Marchamley", "Windermere")
)

threshold_boxplot <- ggplot(plot_df,
  aes(x = type, y = rel_abundance_per_sample, fill = type)
) +
  geom_boxplot(
    alpha = 0.8,
    outlier.shape = 21,
    outlier.size = 2
  ) +
  geom_hline(
    yintercept = 0.001,
    linetype = "dotted",
    colour = "grey40",
    linewidth = 0.7
  ) +
facet_wrap(~dataset, scales = "free_y") +
  scale_x_discrete(
    labels = c(
      false_positive = "False positive",
      menu_or_bait = "Bait or menu taxa",
      true_positive = "True positive"
    )
  ) +
  scale_fill_brewer(palette = "Set2") +
  labs(
    x = "Visual match type",
    y = "Relative abundance per sample (all taxa)",
    fill = "Type"
  ) +
  theme_bw(base_size = 12) +
  theme(
    legend.position = "none",
    strip.background = element_rect(fill = "grey90"),
    strip.text = element_text(face = "bold"),
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank()
  ) +
  ylim(0, 0.0075)

  threshold_boxplot

In [0]:
ggsave(
  filename = "Results/visual_match_relative_abundance_boxplot.png",
  plot = threshold_boxplot,
  width = 10,
  height = 8,
  dpi = 300
)

In [0]:
master_long_worms_df <- master_long_worms_df %>%
  left_join(
    visual_data %>%
      distinct(dataset, taxa_name_clean) %>%
      mutate(visual_match = TRUE),
    by = c(
      "dataset",
      "species" = "taxa_name_clean"
    )
  ) %>%
  mutate(
    visual_match = if_else(is.na(visual_match), FALSE, visual_match)
  )

In [0]:
threshold <- 0.001

species_summary <- master_long_worms_df %>%
  filter(
    visual_match == TRUE,
    rel_abundance_per_sample < threshold
  ) %>%
  group_by(denoise_method, taxonomy_method, dataset, sampleid) %>%
  summarise(
    n_unique_species = n_distinct(species),
    species_list = paste(sort(unique(species)), collapse = ", "),
    .groups = "drop"
  )

write.csv(species_summary, file = "Results/species_underthreshold_summary.csv")

sum(species_summary$n_unique_species)

In [0]:
p<- ggplot(
  master_long_worms_df,
  aes(
    x = sampleid,
    y = rel_abundance_per_sample,
    fill = visual_match
  )
) +
  geom_col(position = "fill", width = 0.9) +
  facet_grid(
    denoise_method ~ taxonomy_method
  ) +
  scale_y_continuous(
    labels = scales::percent_format()
  ) +
  scale_fill_manual(
    values = c(
      "TRUE" = "red",
      "FALSE" = "grey"
    ),
    labels = c(
      "TRUE" = "Visual match",
      "FALSE" = "eDNA only"
    ),
    name = "Detection status"
  ) +
  labs(
    x = "Sample",
    y = "% relative abundance (all taxa)"
  ) +
  theme_bw() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1),
    #axis.ticks.x = element_blank(),
    #panel.grid.major.x = element_blank(),
    strip.background = element_rect(fill = "grey95"),
    legend.position = "top"
  )

p  

In [0]:
ggsave(
  filename = "Results/visual_match_relative_abundance.png",
  plot = p,
  width = 25,
  height = 18,
  dpi = 300
)

In [0]:
master_long_df_filtered <- master_long_worms_df %>%
  filter(rel_abundance_per_sample >= threshold) %>%
  select(-taxa_reads, -total_reads)

## Create a phyloseq object

In [0]:
source("Scripts/10c_create_phyloseq.R")

## Assign functional groups

We now add if certain taxa are fish or non-fish to both long and phyloseq data.

In [0]:
fish_classes <- c(
  "Teleostei",      # most bony fishes
  "Chondrostei",    # sturgeons, paddlefish
  "Elasmobranchii", # sharks, rays, skates
  "Holocephali",     # chimaeras
  "Petromyzontida" # lampreys
)

In [0]:
tax <- as.data.frame(tax_table(phylo_eDNA))

tax$fun_group <- ifelse(
  tax$class %in% fish_classes,
  "fish",
  "non_fish"
)

tax_table(phylo_eDNA) <- tax_table(as.matrix(tax))

In [0]:
master_long_df_filtered$fun_group <- ifelse(
  master_long_df_filtered$class %in% fish_classes,
  "fish",
  "non_fish"
)

In [0]:
table(master_long_df_filtered$class,
      master_long_df_filtered$fun_group,
      useNA = "ifany")

### Subset data by functional group

In [0]:
# master
master_long_worms_df_fish <- master_long_df_filtered %>% subset(fun_group == "fish") # fish
master_long_worms_df_nonfish <- master_long_df_filtered %>% subset(fun_group != "fish") # non fish

# phyloseq
phylo_eDNA_fish <- subset_taxa(phylo_eDNA, fun_group == "fish")
phylo_eDNA_nonfish <- subset_taxa(phylo_eDNA, fun_group != "fish")

In [0]:
windermere_fish <- master_long_worms_df_fish %>%
filter(dataset == "windermere_2017", fun_group == "fish")

windermere_fish_list <- windermere_fish %>%
distinct(species, genus, family, class, order)

# save
write.csv(windermere_fish_list, file = "Results/Windermere_2017/unique_species_list.csv")

## Save final datasets

In [0]:
#overall (filtered)
write.csv(master_long_df_filtered, file = "Data/Processed/master_long_worms_df_filtered.csv")
saveRDS(phylo_eDNA, "Data/Processed/phylo_eDNA.RDS")

# overall (before filtering)
write.csv(master_long_worms_df, file = "Data/Processed/master_long_worms_df.csv")

#fish
write.csv(master_long_worms_df_fish, file = "Data/Processed/master_long_worms_df_filtered_fish.csv")
saveRDS(phylo_eDNA_fish, "Data/Processed/phylo_eDNA_fish.RDS")

#non-fish
write.csv(master_long_worms_df_nonfish, file = "Data/Processed/master_long_worms_df_filtered_nonfish.csv")
saveRDS(phylo_eDNA_nonfish, "Data/Processed/phylo_eDNA_nonfish.RDS")